# <center>Preprocesamiento Mezcla i3 + tree_safe</center>


Este notebook arma una variante hibrida entre `i3_arboles` y nuestro `tree_safe`. La idea es conservar el feature engineering fisico de `i3`, pero aplicar el manejo metodologicamente mas prolijo de `tree_safe`: `-999 -> NaN` al cargar, split `train/valid` antes de ajustar cualquier transformacion, one-hot de `PRI_jet_num` ajustado solo con train, indicadores de missing y guardado del pipeline ajustado.


## Imports y utilidades


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_mix_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        direct_candidate = candidate / "i3_tree_safe_mix.py"
        nested_candidate = candidate / "11.Mezcla" / "i3_tree_safe_mix.py"
        if direct_candidate.exists():
            return candidate
        if nested_candidate.exists():
            return candidate / "11.Mezcla"
    raise FileNotFoundError("No se pudo ubicar 11.Mezcla/i3_tree_safe_mix.py")


mix_dir = find_mix_dir()
if str(mix_dir) not in sys.path:
    sys.path.insert(0, str(mix_dir))

from i3_tree_safe_mix import (
    get_higgs_data_dir,
    run_i3_tree_safe_mix,
    save_i3_tree_safe_mix_artifacts,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)


## Que mezcla esta variante
- De `i3`: ratios fisicos, `HT`, `visible_energy`, distancias angulares, interacciones, cuadrados y ranks.
- De `tree_safe`: `NaN` conservados, sin imputacion, sin escalado, flags `is_missing_*`, split temprano y pipeline serializado.
- Diferencia clave respecto a `i3_arboles`: los ranks se ajustan solo con train y luego se aplican a valid/test usando la distribucion empirica aprendida.


In [2]:
artifacts = run_i3_tree_safe_mix()
output_dir = get_higgs_data_dir() / "processed" / "i3_tree_safe_mix"
metadata = artifacts["metadata"]

print("Variante: Mezcla i3 + tree_safe")
print(f"Directorio de salida esperado: {output_dir}")
print("Descripcion:")
print(metadata["description"])


Variante: Mezcla i3 + tree_safe
Directorio de salida esperado: /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_safe_mix
Descripcion:
Features fisicas inspiradas en i3, ranks ajustados solo con train, NaN conservados, missing flags, split temprano y one-hot de PRI_jet_num ajustado solo sobre train.


## Resumen del procesamiento


In [3]:
print("Resumen del split temprano:")
display(artifacts["split_summary_df"].round(4))

print("Missingness en train crudo luego del reemplazo -999 -> NaN:")
display(artifacts["train_missing_summary_df"].head(15))

print("Metadata principal del procesamiento:")
for key, value in metadata.items():
    print(f"- {key}: {value}")

print("Primeras filas del train procesado:")
display(artifacts["X_train_processed"].head())
print("Etiquetas de train:")
display(artifacts["y_train"].head().to_frame(name="Label"))

print("Features nuevas agregadas por la mezcla:")
display(pd.DataFrame({"feature": artifacts["engineered_feature_columns"]}).head(40))

print("Columnas base sobre las que se calcularon ranks train-only:")
display(pd.DataFrame({"feature": artifacts["rank_source_columns"]}))

print("Columnas con flags de missing:")
display(pd.DataFrame({"feature": artifacts["missing_indicator_source_columns"]}).head(30))


Resumen del split temprano:


,split,rows,class_b,class_s,weight_sum
0,train,225000,147900.0,77100.0,370822.7398
1,valid,25000,16433.0,8567.0,40869.0961
2,test_external,550000,NaN,NaN,NaN


Missingness en train crudo luego del reemplazo -999 -> NaN:


,feature,missing_count,missing_pct
0,DER_deltaeta_jet_jet,159718,70.99
1,DER_lep_eta_centrality,159718,70.99
2,DER_mass_jet_jet,159718,70.99
3,DER_prodeta_jet_jet,159718,70.99
4,PRI_jet_subleading_eta,159718,70.99
5,PRI_jet_subleading_phi,159718,70.99
6,PRI_jet_subleading_pt,159718,70.99
7,PRI_jet_leading_eta,89987,39.99
8,PRI_jet_leading_phi,89987,39.99
9,PRI_jet_leading_pt,89987,39.99


Metadata principal del procesamiento:
- variant_key: i3_tree_safe_mix
- variant_title: Mezcla i3 + tree_safe
- description: Features fisicas inspiradas en i3, ranks ajustados solo con train, NaN conservados, missing flags, split temprano y one-hot de PRI_jet_num ajustado solo sobre train.
- valid_size: 0.1
- random_state: 42
- raw_feature_count: 30
- engineered_feature_count: 33
- processed_feature_count: 81
- rank_feature_count: 7
- missing_indicator_count: 15
- processed_train_nan_count: 1615278
- processed_valid_nan_count: 179029
- processed_test_nan_count: 3945016
Primeras filas del train procesado:


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,DER_pt_ratio_lep_tau,DER_met_phi_centrality,DER_lep_eta_centrality,PRI_tau_pt,PRI_tau_eta,PRI_tau_phi,PRI_lep_pt,PRI_lep_eta,PRI_lep_phi,PRI_met,PRI_met_phi,PRI_met_sumet,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,ratio_tau_lep_pt,ratio_tau_met,ratio_lep_met,ratio_mass_pt,jet_lep_ratio,HT,visible_energy,met_fraction,abs_eta_diff_tau_lep,delta_phi_tau_lep,abs_phi_diff_tau_lep,deltaR_tau_lep,jets_pt_sum,jet_lepton_balance,tau_lep_interaction,tau_met_interaction,has_jet,ht_if_jets,PRI_tau_pt_sq,PRI_lep_pt_sq,PRI_met_sq,DER_mass_MMC_sq,PRI_jet_leading_pt_sq,HT_sq,jets_pt_sum_sq,deltaR_tau_lep_sq,DER_mass_MMC_rank,PRI_tau_pt_rank,PRI_lep_pt_rank,PRI_met_rank,HT_rank,deltaR_tau_lep_rank,jets_pt_sum_rank,is_missing_DER_mass_MMC,is_missing_DER_deltaeta_jet_jet,is_missing_DER_mass_jet_jet,is_missing_DER_prodeta_jet_jet,is_missing_DER_lep_eta_centrality,is_missing_PRI_jet_leading_pt,is_missing_PRI_jet_leading_eta,is_missing_PRI_jet_leading_phi,is_missing_PRI_jet_subleading_pt,is_missing_PRI_jet_subleading_eta,is_missing_PRI_jet_subleading_phi,is_missing_ratio_mass_pt,is_missing_DER_mass_MMC_sq,is_missing_PRI_jet_leading_pt_sq,is_missing_DER_mass_MMC_rank,PRI_jet_num_0,PRI_jet_num_1,PRI_jet_num_2,PRI_jet_num_3
0,109.492996,40.445999,81.954002,37.280998,NaN,NaN,NaN,2.906,37.280998,86.779999,0.525,-1.406,NaN,56.905998,0.401,2.182,29.874001,0.324,-0.723,15.662000,-3.139,155.645004,NaN,NaN,NaN,NaN,NaN,NaN,-0.000000,1.904867,3.633380,1.907419,1.261731,0.000000,102.442001,86.779999,0.152887,0.077,2.905000,2.905000,2.906020,0.000000,0.000000,1700.009888,891.261780,0.0,0.000000,3238.292725,892.455872,245.298248,11988.716797,NaN,10494.363281,0.000000,8.444954,0.465475,0.865133,0.156227,0.150240,0.194293,0.716053,0.399942,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,NaN,71.706001,104.612999,1.741000,NaN,NaN,NaN,2.618,1.741000,93.132004,1.041,-1.413,NaN,45.625000,0.830,-2.159,47.507000,2.054,1.810,36.578999,-0.261,115.133003,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.960385,1.247300,1.298751,NaN,0.000000,129.710999,93.132004,0.282004,1.224,2.314185,2.314185,2.617944,0.000000,0.000000,2167.506836,1668.916870,0.0,0.000000,2081.640625,2256.915039,1338.023193,NaN,NaN,16824.943359,0.000000,6.853630,NaN,0.758404,0.655049,0.530982,0.376591,0.558844,0.399942,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
2,106.320000,7.259000,57.109001,91.891998,NaN,NaN,NaN,1.971,25.091000,176.307999,1.003,1.154,NaN,31.552000,2.059,2.421,31.656000,1.125,-2.127,56.681000,-2.298,309.502014,113.099998,-2.667,0.686,NaN,NaN,NaN,113.099998,0.996715,0.556659,0.558494,1.682065,3.572782,232.988998,63.208000,0.243278,0.934,-1.735185,1.735185,1.970590,113.099998,3.572782,998.810120,1788.398926,1.0,232.988998,995.528687,1002.102356,3212.735840,11303.942383,12791.610352,54283.875000,12791.610352,3.883224,0.426644,0.493133,0.223804,0.796578,0.735107,0.300729,0.772547,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,NaN,81.124001,91.842003,46.893002,NaN,NaN,NaN,2.862,1.888000,127.359001,0.761,-1.408,NaN,45.945999,0.765,-1.780,34.952999,1.932,0.833,60.183998,3.003,150.822006,46.459000,1.779,0.271,NaN,NaN,NaN,46.459000,1.314508,0.763425,0.580769,NaN,1.329185,187.542007,80.899002,0.320909,1.167,-2.613000,2.613000,2.861758,46.459000,1.329185,1605.950562,2765.214111,1.0,187.542007,2111.034912,1221.712158,3622.113770,NaN,2158.438721,35172.000000,2158.438721,8.189658,NaN,0.762458,0.337933,0.822862,0.614436,0.689613,0.537680,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
4,88.685997,46.839001,59.231998,121.435997,3.813,637.747009,-3.345,1.614,13.441000,296.751007,2.641,1.378,0.288,23.315001,-0.651,-1.853,61.580002,-1.587,-0.539,

... (4 outputs left out)

Columnas base sobre las que se calcularon ranks train-only:


,feature
0,DER_mass_MMC
1,PRI_tau_pt
2,PRI_lep_pt
3,PRI_met
4,HT
5,deltaR_tau_lep
6,jets_pt_sum


Columnas con flags de missing:


,feature
0,DER_mass_MMC
1,DER_deltaeta_jet_jet
2,DER_mass_jet_jet
3,DER_prodeta_jet_jet
4,DER_lep_eta_centrality
5,PRI_jet_leading_pt
6,PRI_jet_leading_eta
7,PRI_jet_leading_phi
8,PRI_jet_subleading_pt
9,PRI_jet_subleading_eta


## Guardado de artefactos
Se guardan los `csv` procesados del split train/valid/test, la metadata del split, los nombres de features, el pipeline ajustado y las tablas auxiliares.


In [4]:
saved_paths = save_i3_tree_safe_mix_artifacts(artifacts, output_dir=output_dir)

print(f"Directorio de salida: {output_dir}")
print(f"Cantidad de artefactos guardados: {len(saved_paths)}")
for path in saved_paths:
    print(f"- {path}")


Directorio de salida: /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_safe_mix
Cantidad de artefactos guardados: 21
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_safe_mix/X_train_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_safe_mix/X_valid_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_safe_mix/X_test_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_safe_mix/y_train.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_safe_mix/y_valid.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_safe_mix/train_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_safe_mix/valid_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_safe_mix/train_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i3_tree_